# Step 32 — Build waveform-defined and pick-only candidate events

Run after steps 10, 22 and 30. The workflow is implemented below; shared normalization/time helpers remain imported. It writes event, per-pick association and QC CSVs under the configured external output root. These are archival association containers, not a relocated or scientifically validated earthquake catalogue.

## Algorithm and meanings

1. Read the waveform index, merged picks and step-30 map. Reject map references to missing pick IDs and conflicting multi-waveform mappings. Invalid waveform times are excluded; review the input index for these cases.
2. Create one container per indexed waveform (`wav_<waveform_id>`). Attach primary picks via step 30. Reconstruct their timestamps as waveform start plus the step-30 offset: this reproduces the earlier pick time rather than correcting its clock.
3. Attach secondary picks individually when their recorded times lie inside the file-wide waveform window expanded by `time_tolerance` (0.5 s in this notebook). Picks already assigned are excluded. **If windows overlap, the first waveform in input order claims an eligible secondary pick**; no best-match score or ambiguity search is performed. A monthly source group can therefore be split across containers.
4. Apply event-local duplicate checks by normalized SEED ID/phase and time tolerance. Remaining picks, including event-local drops that were not marked assigned, enter pick-only groups keyed by their original source event ID. Check the output pick-ID conservation rather than assuming every pick-only row represents a missing recording.
5. Classify each container: `WAV+PICKS` has a waveform and at least one candidate pick; `WAV_ONLY` has no assigned picks; `PICKS_ONLY` contains remaining picks without an assigned waveform. A single earthquake may be represented by multiple containers. There is no hypocenter association here.

## What this does not validate

Time inclusion is not a channel-identity check, signal check or clock calibration. Missing channels, picks outside a channel's shorter window, wrong phase labels, partial DMX recovery and overlapping triggered windows remain review cases. Preserve raw-line/source IDs and inspect waveform-plus-pick plots after association. Input priority is a preference rule, not a statement that every primary pick is correct.

The QC file includes per-container counts and a run-summary row. `assigned_unique_pick_ids` in that summary includes pick-only containers; it is not the number of picks linked to waveforms. See step 34 for the saved-run review.

## Multiple picked events and PI2L overlay policy (11 November 2026)

A triggered waveform file can contain several earthquakes. `event_id=wav_...` is a waveform container, not proof of a single earthquake. Preserve `pick_group_id` and `event_id_source` when plotting; annotate different source groups separately. Distinct group IDs can also represent the same event in different source files, so do not automatically interpret every group as a separate earthquake. Existing time-window association and 0.5-second duplicate rules remain unchanged; arrivals farther apart are retained.

Saved-run examples: waveform `91060317` contains individual groups `91060317` (18:30:30.23–18:30:38.28) and `910603A7` (18:31:04.45–18:31:11.27). Waveform `9106031E` contains groups `9106031E` and `910603AE` at 19:52:34.15–19:53:00.15 and 19:53:07.04–19:53:22.59. These recorded-time clusters are strong candidates for multiple picked events; confirm the signals in waveform review.

PI2L picks retain their source identity (`seed_id`, `seed_norm`, `original_seed_id`) and timestamps. The archive's current low-gain mapping is **ELZ**, not LHZ; the overlay rule accepts both labels without changing source metadata. The new `waveform_seed_id` explicitly targets `XB.PI2..EHZ` for low-gain picks when PI2Z is present in the indexed waveform and there are no existing PI2Z picks anywhere in that file. `include_waveform_overlay` and `waveform_overlay_rule` record whether and why a marker should be displayed. If PI2Z picks already exist, omit low-gain overlays for the entire file, preserving their rows for audit. This is the requested conservative file-wide rule; it may omit a PI2L pick from another earthquake or phase in the same file. Revisit using group/phase/time-aware suppression after waveform inspection if needed.

This is an **annotation policy**, not a new gain calibration, recovered low-gain trace, timestamp correction or scientific reidentification of the arrival. Placing a low-gain marker on the normal-gain waveform does not validate the pick. The index's separate station/channel lists are only a preliminary target screen: a future plotting routine must require the exact `waveform_seed_id` trace and check that its time span contains the pick. Other absent-channel picks still need their own review.

Downstream waveform plotting must read `waveform_seed_id` and honor `include_waveform_overlay`. Existing scientific catalogue exporters continue to use the original pick identity; these new columns do not silently change those exporters. Rerun step 32 to populate the new columns; no production outputs are updated merely by opening this notebook.


## 1. Configuration

Inspect source paths, output paths and tolerances. This cell resolves configuration without running association or plotting.

In [1]:
from pathlib import Path
import sys
from IPython.display import display

# Find the adjacent local package whether Jupyter starts here or in the project root.
_candidates = [Path.cwd(), Path.cwd() / '10_conversion', *Path.cwd().parents]
PIPELINE_DIR = next((p for p in _candidates if (p / 'pinatubo_steps').is_dir()), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError('Open this notebook from the pipeline folder or project root.')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))
from pinatubo_steps.config import pipeline_paths

CONFIG_FILE = None  # Default: conversion_config.json; or set an explicit JSON path.
P = pipeline_paths(config_file=CONFIG_FILE)
DATA_TOP = Path(P['DATA_TOP'])
print('Configuration:', P['CONFIG_FILE'])
print('External outputs:', P['OUTPUT_ROOT'])

SETTINGS = dict(
    waveform_index=P['WAVEFORM_INDEX'],
    merged_picks=P['MERGED_PICK_INDEX'],
    individual_pick_waveform_map=P['INDIV_PICK_WAVEFORM_MAP'],
    out_event_csv=P['WAVEFORM_PICK_EVENT_INDEX'],
    out_pick_map_csv=P['PICK_MAP_CSV'],
    out_qc_csv=P['QC_CSV'],
    time_tolerance=0.5,
)
display(SETTINGS)

Configuration: /Users/thompsong/Developer/Pinatubo1991SeismicData/10_conversion/conversion_config.json
External outputs: /Users/thompsong/work/Pinatubo1991Processing/converted


{'waveform_index': '/Users/thompsong/work/Pinatubo1991Processing/converted/pipeline_tmp/10_waveform_index.csv',
 'merged_picks': '/Users/thompsong/work/Pinatubo1991Processing/converted/pipeline_tmp/22_merged_pick_index.csv',
 'individual_pick_waveform_map': '/Users/thompsong/work/Pinatubo1991Processing/converted/pipeline_tmp/30_individual_pick_waveform_map.csv',
 'out_event_csv': '/Users/thompsong/work/Pinatubo1991Processing/converted/pipeline_tmp/32_waveform_pick_event_index.csv',
 'out_pick_map_csv': '/Users/thompsong/work/Pinatubo1991Processing/converted/pipeline_tmp/32_waveform_pick_event_map.csv',
 'out_qc_csv': '/Users/thompsong/work/Pinatubo1991Processing/converted/pipeline_tmp/qc/32_event_catalog_qc.csv',
 'time_tolerance': 0.5}

## 2. Implementation

These definitions make the workflow inspectable here. Defining them does not write files; the run cell below executes them.

In [2]:
from __future__ import annotations

from pathlib import Path

from typing import Dict, Tuple, Optional, List

import pandas as pd

from pinatubo_steps.shared import (to_dt, ensure_cols, normalize_seed_id_df)

def validate_step30_map(df_map: pd.DataFrame, df_picks: pd.DataFrame) -> None:
    """
    Sanity checks:
      - All pick_id values in Step 30 must exist in Step 22
      - No pick_id appears multiple times with different waveform_id
    """
    missing = set(df_map["pick_id"]) - set(df_picks["pick_id"])
    if missing:
        raise ValueError(
            f"STEP 30 map contains pick_id values not present in STEP 22: "
            f"{sorted(list(missing))[:10]} ..."
        )

    dup = (
        df_map.groupby("pick_id")["waveform_id"]
        .nunique()
        .reset_index()
        .query("waveform_id > 1")
    )
    if not dup.empty:
        raise ValueError(
            "STEP 30 map assigns the same pick_id to multiple waveforms. "
            "This violates analyst-pick assumptions."
        )

def infer_pick_priority(df: pd.DataFrame) -> pd.Series:
    """
    Prefer existing pick_priority if provided by STEP 22.
    Otherwise infer from event_id_source / provenance.
    """
    if "pick_priority" in df.columns:
        s = df["pick_priority"].astype("string").fillna("").str.lower().str.strip()
        # normalize variants
        s = s.replace({"prim": "primary", "sec": "secondary"})
        # fallback inference where blank
        blank = s.eq("")
        if blank.any():
            src = df.get("event_id_source", pd.Series([""] * len(df), index=df.index)).astype("string").fillna("").str.lower()
            s = s.mask(blank & src.str.contains("individual"), "primary")
            s = s.mask(blank & ~src.str.contains("individual"), "secondary")
        return s

    src = df.get("event_id_source", pd.Series([""] * len(df), index=df.index)).astype("string").fillna("").str.lower()
    out = pd.Series(["secondary"] * len(df), index=df.index, dtype="string")
    out = out.mask(src.str.contains("individual"), "primary")
    return out

def dedupe_picks_within_event(
    df: pd.DataFrame,
    *,
    time_col: str,
    tol: pd.Timedelta,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Event-local dedupe by (seed_norm, phase) within tolerance.
    Keeps first occurrence in sort order.
    """
    if df.empty:
        return df.copy(), df.iloc[0:0].copy()

    df = df.copy()

    # Ensure keys exist
    df = ensure_cols(df, ["seed_norm", "phase"])
    df["phase"] = df["phase"].astype("string")

    # Stable ordering: primary first, then by time
    if "pick_priority" in df.columns:
        pri = df["pick_priority"].astype("string").fillna("")
        # primary sorts before secondary
        pri_rank = pri.map({"primary": 0, "secondary": 1}).fillna(2).astype(int)
        df["_pri_rank"] = pri_rank
        df = df.sort_values(["_pri_rank", "seed_norm", "phase", time_col]).reset_index(drop=True)
    else:
        df = df.sort_values(["seed_norm", "phase", time_col]).reset_index(drop=True)

    key_cols = ["seed_norm", "phase"]
    df["_dt"] = df.groupby(key_cols)[time_col].diff()

    dup_mask = df["_dt"].notna() & (df["_dt"].abs() <= tol)

    dropped = df.loc[dup_mask].drop(columns=["_dt", "_pri_rank"], errors="ignore")
    kept = df.loc[~dup_mask].drop(columns=["_dt", "_pri_rank"], errors="ignore")

    return kept.reset_index(drop=True), dropped.reset_index(drop=True)

def add_waveform_overlay_targets(pick_map: pd.DataFrame, waveforms: pd.DataFrame) -> pd.DataFrame:
    """Keep source IDs; choose explicit annotation targets for PI2 low-gain picks.

    This affects future waveform review annotations, not the scientific identity
    of the source pick or the event association. Do not use overlay targets as
    instrument-response metadata or evidence of an acquired low-gain waveform.
    """
    out = pick_map.copy()
    out['original_seed_id'] = out['seed_id']
    linked = out['event_class'].eq('WAV+PICKS')
    out['include_waveform_overlay'] = linked
    out['waveform_seed_id'] = out['seed_norm'].where(linked, pd.NA)
    out['waveform_overlay_rule'] = 'original_identity'
    out.loc[~linked, 'waveform_overlay_rule'] = 'no_associated_waveform'
    low = linked & out['seed_norm'].isin(['XB.PI2..ELZ', 'XB.PI2..LHZ'])
    high = linked & out['seed_norm'].eq('XB.PI2..EHZ')
    high_files = set(out.loc[high, 'waveform_id'].dropna().astype(str))
    blocked = low & out['waveform_id'].astype(str).isin(high_files)
    out.loc[blocked, 'include_waveform_overlay'] = False
    out.loc[blocked, 'waveform_seed_id'] = pd.NA
    out.loc[blocked, 'waveform_overlay_rule'] = 'pi2l_omitted_existing_pi2z_pick_in_file'
    # Use index membership only as a preliminary target screen. Exact trace ID
    # and pick-time coverage must still be checked when reading the waveform.
    available = set()
    for row in waveforms.itertuples(index=False):
        stations = str(getattr(row, 'stations', '')).split(',')
        channels = str(getattr(row, 'channels', '')).split(',')
        if 'PI2' in [x.strip() for x in stations] and 'EHZ' in [x.strip() for x in channels]:
            available.add(str(row.event_id))
    remap = low & ~blocked & out['waveform_id'].astype(str).isin(available)
    out.loc[remap, 'waveform_seed_id'] = 'XB.PI2..EHZ'
    out.loc[remap, 'waveform_overlay_rule'] = 'pi2l_display_on_pi2z'
    unavailable = low & ~blocked & ~remap
    out.loc[unavailable, 'include_waveform_overlay'] = False
    out.loc[unavailable, 'waveform_seed_id'] = pd.NA
    out.loc[unavailable, 'waveform_overlay_rule'] = 'pi2z_target_not_in_index'
    return out

def build_event_catalog(
    df_wav: pd.DataFrame,
    df_picks: pd.DataFrame,
    df_map: pd.DataFrame,
    *,
    tol_s: float,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    tol = pd.Timedelta(seconds=float(tol_s))

    # -----------------------------
    # Normalize/validate waveform index
    # -----------------------------
    df_wav = df_wav.copy()
    for col in ("event_id", "starttime", "endtime"):
        if col not in df_wav.columns:
            raise ValueError(f"waveform-index missing required column: {col}")

    df_wav["starttime"] = to_dt(df_wav["starttime"])
    df_wav["endtime"] = to_dt(df_wav["endtime"])

    wav_bad = df_wav["starttime"].isna() | df_wav["endtime"].isna()
    if wav_bad.any():
        df_wav = df_wav.loc[~wav_bad].reset_index(drop=True)

    # -----------------------------
    # Normalize/validate merged picks (STEP 22)
    # -----------------------------
    df_picks = df_picks.copy()
    if "pick_id" not in df_picks.columns:
        raise ValueError("merged-picks (Step 22) missing required column: pick_id")
    if "event_id" not in df_picks.columns:
        raise ValueError("merged-picks (Step 22) missing required column: event_id (pick group id)")
    if "pick_time" not in df_picks.columns:
        raise ValueError("merged-picks (Step 22) missing required column: pick_time")

    df_picks["pick_time"] = to_dt(df_picks["pick_time"])
    # Keep NaT picks for QC, but they cannot be assigned by time-window logic
    df_picks["seed_norm"] = normalize_seed_id_df(df_picks)
    df_picks["pick_priority"] = infer_pick_priority(df_picks)

    # Convenience splits
    df_primary = df_picks[df_picks["pick_priority"].astype("string") == "primary"].copy()
    df_secondary = df_picks[df_picks["pick_priority"].astype("string") == "secondary"].copy()

    # -----------------------------
    # Normalize/validate mapping (STEP 30)
    # -----------------------------
    df_map = df_map.copy()
    # Validate STEP 30 mapping integrity
    validate_step30_map(df_map, df_picks)
    for col in ("pick_id", "waveform_id", "time_offset_from_start"):
        if col not in df_map.columns:
            raise ValueError(f"individual-pick-waveform-map (Step 30) missing required column: {col}")

    # Map can (and should) include only primary picks; we won't assume, we'll enforce later.
    # Ensure numeric offsets
    df_map["time_offset_from_start"] = pd.to_numeric(df_map["time_offset_from_start"], errors="coerce")

    # Join mapping -> pick attributes for authoritative individual picks
    ind_full = df_map.merge(
        df_primary,
        on="pick_id",
        how="left",
        validate="many_to_one",
        suffixes=("", "_pick"),
    )

    # Attach waveform start times for reconstruction
    ind_full = ind_full.merge(
        df_wav[["event_id", "starttime", "endtime", "wav_file"]],
        left_on="waveform_id",
        right_on="event_id",
        how="left",
        suffixes=("", "_wav"),
    )

    ind_full["pick_time_reconstructed"] = (
        ind_full["starttime"] + pd.to_timedelta(ind_full["time_offset_from_start"], unit="s")
    )

    # Keep only recon-good and waveform-known (if waveform missing, this mapping is unusable)
    ind_bad = (
        ind_full["pick_time_reconstructed"].isna()
        | ind_full["waveform_id"].isna()
        | ind_full["starttime"].isna()
        | ind_full["endtime"].isna()
    )
    ind_full_good = ind_full.loc[~ind_bad].copy().reset_index(drop=True)

    # Group authoritative picks by waveform_id
    ind_by_wav = (
        ind_full_good.groupby("waveform_id", sort=False)
        if not ind_full_good.empty else None
    )

    # -----------------------------
    # PASS 1: waveform-defined events
    # -----------------------------
    assigned_pick_ids = set()

    events_rows: List[dict] = []
    pick_rows: List[dict] = []
    qc_rows: List[dict] = []

    # Pre-filter secondary picks with valid times only (time-window matching)
    sec_valid_time = df_secondary["pick_time"].notna()
    df_secondary_time = df_secondary.loc[sec_valid_time].copy()

    for _, w in df_wav.iterrows():
        waveform_id = str(w["event_id"])
        start = w["starttime"]
        end = w["endtime"]
        wav_file = w.get("wav_file")

        ev_id = f"wav_{waveform_id}"

        # ---- authoritative picks for this waveform via STEP 30
        if ind_by_wav is not None and waveform_id in ind_by_wav.groups:
            ind_picks = ind_by_wav.get_group(waveform_id).copy()
        else:
            ind_picks = pd.DataFrame()

        # Build authoritative pick table for output mapping
        n_primary = 0
        n_primary_dedup = 0

        if not ind_picks.empty:
            # Ensure keys
            ind_picks = ensure_cols(ind_picks, ["seed_norm", "phase", "raw_line", "raw_lineno", "pha_file"])
            ind_picks["seed_norm"] = ind_picks.get("seed_norm", normalize_seed_id_df(ind_picks))
            ind_picks["pick_priority"] = "primary"

            # event-local dedupe (just in case mapping duplicates exist)
            ind_tmp = ind_picks.rename(columns={"pick_time_reconstructed": "pick_time_evt"}).copy()
            kept, dropped = dedupe_picks_within_event(ind_tmp, time_col="pick_time_evt", tol=tol)
            n_primary = int(len(ind_tmp))
            n_primary_dedup = int(len(dropped))

            ind_keep = kept.copy()
            ind_keep["pick_time_evt"] = ind_keep["pick_time_evt"]  # explicit
        else:
            ind_keep = pd.DataFrame()
            n_primary = 0
            n_primary_dedup = 0

        # Mark assigned primary picks
        if not ind_keep.empty:
            for pid in ind_keep["pick_id"].astype("string").fillna("").tolist():
                if pid:
                    assigned_pick_ids.add(pid)

        # ---- secondary picks in waveform window (± tol), not yet assigned
        mon = df_secondary_time[
            (df_secondary_time["pick_time"] >= (start - tol)) &
            (df_secondary_time["pick_time"] <= (end + tol))
        ].copy()

        # exclude already assigned pick_ids
        if not mon.empty:
            mon["pick_id"] = mon["pick_id"].astype("string")
            mon = mon[~mon["pick_id"].isin(assigned_pick_ids)].copy()

        # event-local dedupe for monthly picks (optional but highly stabilizing)
        n_secondary = int(len(mon))
        n_secondary_dedup = 0
        if not mon.empty:
            mon = ensure_cols(mon, ["seed_norm", "phase"])
            mon["seed_norm"] = mon.get("seed_norm", normalize_seed_id_df(mon))
            mon["pick_priority"] = "secondary"
            mon_tmp = mon.rename(columns={"pick_time": "pick_time_evt"}).copy()
            mon_keep, mon_drop = dedupe_picks_within_event(mon_tmp, time_col="pick_time_evt", tol=tol)
            n_secondary_dedup = int(len(mon_drop))
        else:
            mon_keep = pd.DataFrame()

        # mark assigned secondary picks
        if not mon_keep.empty:
            for pid in mon_keep["pick_id"].astype("string").fillna("").tolist():
                if pid:
                    assigned_pick_ids.add(pid)

        # ---- Emit pick map rows
        n_out_picks = 0

        if not ind_keep.empty:
            for _, p in ind_keep.iterrows():
                pt = p["pick_time_evt"]
                n_out_picks += 1
                pick_rows.append({
                    "event_id": ev_id,
                    "event_class": None,  # filled later
                    "waveform_id": waveform_id,
                    "wav_file": wav_file,
                    "pick_id": p.get("pick_id"),
                    "pick_source": "individual",
                    "pick_priority": "primary",
                    "pick_group_id": p.get("event_id"),  # STEP22 group id (the individual group id)
                    "event_id_source": p.get("event_id_source"),
                    "seed_id": p.get("seed_id"),
                    "seed_norm": p.get("seed_norm"),
                    "phase": p.get("phase"),
                    "pick_time": pd.Timestamp(pt).isoformat() if pd.notna(pt) else None,
                    "time_offset": float((pt - start).total_seconds()) if pd.notna(pt) else None,
                    "raw_line": p.get("raw_line"),
                    "raw_lineno": p.get("raw_lineno"),
                    "pha_file": p.get("pha_file"),
                })

        if not mon_keep.empty:
            for _, p in mon_keep.iterrows():
                pt = p["pick_time_evt"]
                n_out_picks += 1
                pick_rows.append({
                    "event_id": ev_id,
                    "event_class": None,  # filled later
                    "waveform_id": waveform_id,
                    "wav_file": wav_file,
                    "pick_id": p.get("pick_id"),
                    "pick_source": "monthly",
                    "pick_priority": "secondary",
                    "pick_group_id": p.get("event_id"),  # STEP22 group id (monthly block id)
                    "event_id_source": p.get("event_id_source"),
                    "seed_id": p.get("seed_id"),
                    "seed_norm": p.get("seed_norm"),
                    "phase": p.get("phase"),
                    "pick_time": pd.Timestamp(pt).isoformat() if pd.notna(pt) else None,
                    "time_offset": float((pt - start).total_seconds()) if pd.notna(pt) else None,
                    "raw_line": p.get("raw_line"),
                    "raw_lineno": p.get("raw_lineno"),
                    "pha_file": p.get("pha_file"),
                })

        # ---- Event-level row
        has_primary = not ind_keep.empty
        has_secondary = not mon_keep.empty
        has_waveform = True

        if has_primary or has_secondary:
            event_class = "WAV+PICKS"
        else:
            event_class = "WAV_ONLY"

        events_rows.append({
            "event_id": ev_id,
            "event_class": event_class,
            "waveform_id": waveform_id,
            "wav_file": wav_file,
            "starttime": start.isoformat(),
            "endtime": end.isoformat(),
            "has_waveform": True,
            "has_primary_picks": bool(has_primary),
            "has_secondary_picks": bool(has_secondary),
            "n_picks_total": int((len(ind_keep) if not ind_keep.empty else 0) + (len(mon_keep) if not mon_keep.empty else 0)),
            "n_primary_picks": int(len(ind_keep)) if not ind_keep.empty else 0,
            "n_secondary_picks": int(len(mon_keep)) if not mon_keep.empty else 0,
        })

        qc_rows.append({
            "event_id": ev_id,
            "event_class": event_class,
            "waveform_id": waveform_id,
            "has_waveform": True,
            "n_primary_picks_raw": n_primary,
            "n_primary_picks_dedup_dropped": n_primary_dedup,
            "n_secondary_picks_in_window_raw": n_secondary,
            "n_secondary_picks_dedup_dropped": n_secondary_dedup,
            "n_picks_assigned_total": int((len(ind_keep) if not ind_keep.empty else 0) + (len(mon_keep) if not mon_keep.empty else 0)),
            "notes": "",
        })

    # -----------------------------
    # PASS 2: pick-only events (all remaining picks)
    # -----------------------------
    df_picks_all = df_picks.copy()
    df_picks_all["pick_id"] = df_picks_all["pick_id"].astype("string")
    remaining = df_picks_all[~df_picks_all["pick_id"].isin(assigned_pick_ids)].copy()

    # Picks with NaT times cannot be window-matched; they still become PICKS_ONLY groups (QC them)
    # Group key for pick-only events is STEP22.event_id
    if not remaining.empty:
        remaining = ensure_cols(remaining, ["seed_norm", "phase", "raw_line", "raw_lineno", "pha_file"])
        remaining["seed_norm"] = remaining.get("seed_norm", normalize_seed_id_df(remaining))
        remaining["pick_priority"] = infer_pick_priority(remaining)

        for pick_group_id, g in remaining.groupby("event_id", sort=True):
            ev_id = f"pick_{pick_group_id}"

            # compute start/end from pick_time if possible
            g_times = g["pick_time"].dropna()
            if not g_times.empty:
                start = g_times.min()
                end = g_times.max()
                start_s = start.isoformat()
                end_s = end.isoformat()
            else:
                start_s = None
                end_s = None

            # event-level row
            has_primary = (g["pick_priority"].astype("string") == "primary").any()
            has_secondary = (g["pick_priority"].astype("string") == "secondary").any()

            events_rows.append({
                "event_id": ev_id,
                "event_class": "PICKS_ONLY",
                "waveform_id": pd.NA,
                "wav_file": pd.NA,
                "starttime": start_s,
                "endtime": end_s,
                "has_waveform": False,
                "has_primary_picks": bool(has_primary),
                "has_secondary_picks": bool(has_secondary),
                "n_picks_total": int(len(g)),
                "n_primary_picks": int((g["pick_priority"].astype("string") == "primary").sum()),
                "n_secondary_picks": int((g["pick_priority"].astype("string") == "secondary").sum()),
            })

            qc_rows.append({
                "event_id": ev_id,
                "event_class": "PICKS_ONLY",
                "waveform_id": pd.NA,
                "has_waveform": False,
                "n_primary_picks_raw": int((g["pick_priority"].astype("string") == "primary").sum()),
                "n_primary_picks_dedup_dropped": 0,
                "n_secondary_picks_in_window_raw": 0,
                "n_secondary_picks_dedup_dropped": 0,
                "n_picks_assigned_total": int(len(g)),
                "notes": "pick-only event (no waveform match/available)",
            })

            # pick-map rows
            for _, p in g.iterrows():
                pt = p.get("pick_time")
                pick_rows.append({
                    "event_id": ev_id,
                    "event_class": "PICKS_ONLY",
                    "waveform_id": pd.NA,
                    "wav_file": pd.NA,
                    "pick_id": p.get("pick_id"),
                    "pick_source": "individual" if str(p.get("pick_priority", "")).lower() == "primary" else "monthly",
                    "pick_priority": p.get("pick_priority"),
                    "pick_group_id": pick_group_id,
                    "event_id_source": p.get("event_id_source"),
                    "seed_id": p.get("seed_id"),
                    "seed_norm": p.get("seed_norm"),
                    "phase": p.get("phase"),
                    "pick_time": pd.Timestamp(pt).isoformat() if pd.notna(pt) else None,
                    "time_offset": None,
                    "raw_line": p.get("raw_line"),
                    "raw_lineno": p.get("raw_lineno"),
                    "pha_file": p.get("pha_file"),
                })

    # -----------------------------
    # Finalize pick_map event_class for waveform events
    # -----------------------------
    df_events = pd.DataFrame(events_rows)
    df_pickmap = pd.DataFrame(pick_rows)
    df_qc = pd.DataFrame(qc_rows)

    if not df_pickmap.empty and not df_events.empty:
        # Fill event_class for wav_* picks
        ev_class_map = dict(zip(df_events["event_id"].astype(str), df_events["event_class"].astype(str)))
        df_pickmap["event_class"] = df_pickmap["event_id"].astype(str).map(ev_class_map)

    # Sort for determinism
    if not df_events.empty:
        df_events = df_events.sort_values(["event_class", "event_id"]).reset_index(drop=True)
    if not df_pickmap.empty:
        df_pickmap = df_pickmap.sort_values(["event_id", "pick_priority", "seed_norm", "phase", "pick_time"]).reset_index(drop=True)
    if not df_qc.empty:
        df_qc = df_qc.sort_values(["event_class", "event_id"]).reset_index(drop=True)

    # Run-level QC summary row
    total_picks = int(len(df_picks))
    assigned_picks = int(df_pickmap["pick_id"].astype("string").nunique()) if not df_pickmap.empty else 0
    nat_picks = int(df_picks["pick_time"].isna().sum())

    df_qc = pd.concat(
        [
            df_qc,
            pd.DataFrame([{
                "event_id": "__RUN_SUMMARY__",
                "event_class": "",
                "waveform_id": "",
                "has_waveform": "",
                "n_primary_picks_raw": "",
                "n_primary_picks_dedup_dropped": "",
                "n_secondary_picks_in_window_raw": "",
                "n_secondary_picks_dedup_dropped": "",
                "n_picks_assigned_total": "",
                "notes": (
                    f"total_picks={total_picks}; "
                    f"assigned_unique_pick_ids={assigned_picks}; "
                    f"unparseable_pick_time_rows={nat_picks}; "
                    f"waveform_events={int((df_events['event_class'].str.startswith('WAV')).sum() if not df_events.empty else 0)}; "
                    f"pick_only_events={int((df_events['event_class'] == 'PICKS_ONLY').sum() if not df_events.empty else 0)}"
                ),
            }]),
        ],
        ignore_index=True,
    )

    df_pickmap = add_waveform_overlay_targets(df_pickmap, df_wav)
    return df_events, df_pickmap, df_qc

def run(
    *,
    waveform_index,
    merged_picks,
    individual_pick_waveform_map,
    out_event_csv,
    out_pick_map_csv,
    out_qc_csv,
    time_tolerance=0.5
):
    """Run this Pinatubo step with explicit notebook arguments. Return tables and results for inspection."""
    from types import SimpleNamespace
    args = SimpleNamespace(
        waveform_index=waveform_index,
        merged_picks=merged_picks,
        individual_pick_waveform_map=individual_pick_waveform_map,
        out_event_csv=out_event_csv,
        out_pick_map_csv=out_pick_map_csv,
        out_qc_csv=out_qc_csv,
        time_tolerance=time_tolerance,
    )

    df_wav = pd.read_csv(args.waveform_index, dtype={"event_id": str, "waveform_id": str, "waveform_event_id": str, "source_event_id": str, "pick_id": str, "origin_id": str})
    df_picks = pd.read_csv(args.merged_picks, dtype={"event_id": str, "waveform_id": str, "waveform_event_id": str, "source_event_id": str, "pick_id": str, "origin_id": str})
    df_map = pd.read_csv(args.individual_pick_waveform_map, dtype={"event_id": str, "waveform_id": str, "waveform_event_id": str, "source_event_id": str, "pick_id": str, "origin_id": str})

    df_events, df_pickmap, df_qc = build_event_catalog(
        df_wav, df_picks, df_map, tol_s=float(args.time_tolerance)
    )

    out_event = Path(args.out_event_csv)
    out_pick = Path(args.out_pick_map_csv)
    out_qc = Path(args.out_qc_csv)

    out_event.parent.mkdir(parents=True, exist_ok=True)
    out_pick.parent.mkdir(parents=True, exist_ok=True)
    out_qc.parent.mkdir(parents=True, exist_ok=True)

    df_events.to_csv(out_event, index=False)
    df_pickmap.to_csv(out_pick, index=False)
    df_qc.to_csv(out_qc, index=False)

    # Console summary
    wav_only = int((df_events["event_class"] == "WAV_ONLY").sum()) if not df_events.empty else 0
    wav_picks = int((df_events["event_class"] == "WAV+PICKS").sum()) if not df_events.empty else 0
    picks_only = int((df_events["event_class"] == "PICKS_ONLY").sum()) if not df_events.empty else 0

    print("\n=== STEP 32 COMPLETE ===")
    print(f"Time tolerance (s):     {float(args.time_tolerance)}")
    print(f"Events:                 {len(df_events)}")
    print(f"  WAV_ONLY:             {wav_only}")
    print(f"  WAV+PICKS:            {wav_picks}")
    print(f"  PICKS_ONLY:           {picks_only}")
    print(f"Pick mappings (rows):   {len(df_pickmap)}")
    print(f"Event CSV:              {out_event}")
    print(f"Pick map CSV:           {out_pick}")
    print(f"QC CSV:                 {out_qc}")
    return {k: v for k, v in locals().items() if (k.startswith("df") or k.startswith("catalog") or k.endswith("_rows") or k in ("events", "annotations", "errors", "stats", "diag", "merged", "suppressed", "matches", "rows", "unparsed"))}

## 3. Run

This cell reads the configured inputs and replaces configured outputs. Review the summary before continuing.

In [3]:
result = run(**SETTINGS)

/var/folders/sz/wyydztsx6q7gbrq8b1pl4fl00000gn/T/ipykernel_34144/3147115092.py:564: DtypeWarning: Columns (0: pha_path) have mixed types. Specify dtype option on import or set low_memory=False.
  df_picks = pd.read_csv(args.merged_picks, dtype={"event_id": str, "waveform_id": str, "waveform_event_id": str, "source_event_id": str, "pick_id": str, "origin_id": str})



=== STEP 32 COMPLETE ===
Time tolerance (s):     0.5
Events:                 26774
  WAV_ONLY:             18524
  WAV+PICKS:            2925
  PICKS_ONLY:           5325
Pick mappings (rows):   57780
Event CSV:              /Users/thompsong/work/Pinatubo1991Processing/converted/pipeline_tmp/32_waveform_pick_event_index.csv
Pick map CSV:           /Users/thompsong/work/Pinatubo1991Processing/converted/pipeline_tmp/32_waveform_pick_event_map.csv
QC CSV:                 /Users/thompsong/work/Pinatubo1991Processing/converted/pipeline_tmp/qc/32_event_catalog_qc.csv


In [4]:
# Preview returned tables and summary objects without reloading output files.
for name, value in result.items():
    print(name)
    display(value.head() if hasattr(value, 'head') else
            value[:5] if isinstance(value, list) else value)


df_wav


,event_id,dmx_file,wav_file,starttime,endtime,stations,channels,ntraces,status
0,9105010W,/Users/thompsong/Library/CloudStorage/Box-Box/...,/Users/thompsong/work/Pinatubo1991Processing/c...,1991-05-01T03:10:09.950000,1991-05-01T03:10:50.900000,"PIE,UBO",EHZ,2,existing
1,9105010X,/Users/thompsong/Library/CloudStorage/Box-Box/...,/Users/thompsong/work/Pinatubo1991Processing/c...,1991-05-01T03:39:42.521000,1991-05-01T03:40:59.311000,"PIE,UBO",EHZ,2,existing
2,91050118,/Users/thompsong/Library/CloudStorage/Box-Box/...,/Users/thompsong/work/Pinatubo1991Processing/c...,1991-05-01T10:45:26.647000,1991-05-01T10:46:12.717000,"CAB,PIE,UBO",EHZ,3,existing
3,91050119,/Users/thompsong/Library/CloudStorage/Box-Box/...,/Users/thompsong/work/Pinatubo1991Processing/c...,1991-05-01T10:49:40.597000,1991-05-01T10:50:26.667000,"CAB,PIE,UBO",EHZ,3,existing
4,9105011A,/Users/thompsong/Library/CloudStorage/Box-Box/...,/Users/thompsong/work/Pinatubo1991Processing/c...,1991-05-01T11:20:14.116000,1991-05-01T11:21:00.186000,"CAB,PIE,UBO",EHZ,3,existing


df_picks


,raw_line,raw_lineno,event_id,event_id_source,pick_id,pha_file,pha_path,station,channel,seed_id,phase,pick_time,onset,first_motion,weight,pick_priority
0,BUGZ PU0 910603073938.49 39.99 S 2 ...,10,91060300,individual_pha_filename,indpha_91060300_1,91060300.PHA,/Users/thompsong/Library/CloudStorage/Box-Box/...,BUG,EHZ,XB.BUG..EHZ,P,1991-06-03T07:39:38.490000Z,NaN,U,0.0,primary
1,BUGZ PU0 910603073938.49 39.99 S 2 ...,10,91060300,individual_pha_filename,indpha_91060300_2,91060300.PHA,/Users/thompsong/Library/CloudStorage/Box-Box/...,BUG,EHZ,XB.BUG..EHZ,S,1991-06-03T07:39:39.990000Z,NaN,NaN,2.0,primary
2,BURZ P 4 910603073938.05,7,91060300,individual_pha_filename,indpha_91060300_3,91060300.PHA,/Users/thompsong/Library/CloudStorage/Box-Box/...,BUR,EHZ,XB.BUR..EHZ,P,1991-06-03T07:39:38.050000Z,NaN,NaN,4.0,primary
3,CABZ P 5 910603073953.23,9,91060300,individual_pha_filename,indpha_91060300_4,91060300.PHA,/Users/thompsong/Library/CloudStorage/Box-Box/...,CAB,EHZ,XB.CAB..EHZ,P,1991-06-03T07:39:53.230000Z,NaN,NaN,5.0,primary
4,GRNZ P 5 910603073953.23,2,91060300,individual_pha_filename,indpha_91060300_5,91060300.PHA,/Users/thompsong/Library/CloudStorage/Box-Box/...,GRN,EHZ,XB.GRN..EHZ,P,1991-06-03T07:39:53.230000Z,NaN,NaN,5.0,primary


df_map


,pick_id,waveform_id,waveform_event_id,time_offset_from_start,raw_line,raw_lineno,pha_file
0,indpha_91060300_1,91060300,91060300,10.489,BUGZ PU0 910603073938.49 39.99 S 2 ...,10,91060300.PHA
1,indpha_91060300_2,91060300,91060300,11.989,BUGZ PU0 910603073938.49 39.99 S 2 ...,10,91060300.PHA
2,indpha_91060300_3,91060300,91060300,10.049,BURZ P 4 910603073938.05,7,91060300.PHA
3,indpha_91060300_4,91060300,91060300,25.229,CABZ P 5 910603073953.23,9,91060300.PHA
4,indpha_91060300_5,91060300,91060300,25.229,GRNZ P 5 910603073953.23,2,91060300.PHA


... (2 outputs left out)

df_pickmap


,event_id,event_class,waveform_id,wav_file,pick_id,pick_source,pick_priority,pick_group_id,event_id_source,seed_id,...,phase,pick_time,time_offset,raw_line,raw_lineno,pha_file,original_seed_id,include_waveform_overlay,waveform_seed_id,waveform_overlay_rule
0,pick_91060310,PICKS_ONLY,NaN,NaN,indpha_91060310_334,individual,primary,91060310,individual_pha_filename,XB.BUG..EHZ,...,P,1991-06-03T17:17:48.670000+00:00,NaN,BUGZ PD0 910603171748.67 ...,10,91060310.PHA,XB.BUG..EHZ,False,NaN,no_associated_waveform
1,pick_91060310,PICKS_ONLY,NaN,NaN,indpha_91060310_335,individual,primary,91060310,individual_pha_filename,XB.BUR..EHZ,...,P,1991-06-03T17:17:49.170000+00:00,NaN,BURZ P 3 910603171749.17,7,91060310.PHA,XB.BUR..EHZ,False,NaN,no_associated_waveform
2,pick_91060310,PICKS_ONLY,NaN,NaN,indpha_91060310_336,individual,primary,91060310,individual_pha_filename,XB.CAB..EHZ,...,P,1991-06-03T17:17:57.960000+00:00,NaN,CABZ P 5 910603171757.96,9,91060310.PHA,XB.CAB..EHZ,False,NaN,no_associated_waveform
3,pick_91060310,PICKS_ONLY,NaN,NaN,indpha_91060310_337,individual,primary,91060310,individual_pha_filename,XB.GRN..EHZ,...,P,1991-06-03T17:17:48.230000+00:00,NaN,GRNZ P 4 910603171748.23 48.22 S 3,2,91060310.PHA,XB.GRN..EHZ,False,NaN,no_associated_waveform
4,pick_91060310,PICKS_ONLY,NaN,NaN,indpha_91060310_338,individual,primary,91060310,individual_pha_filename,XB.GRN..EHZ,...,S,1991-06-03T17:18:48.220000+00:00,NaN,GRNZ P 4 910603171748.23 48.22 S 3,2,91060310.PHA,XB.GRN..EHZ,False,NaN,no_associated_waveform


df_qc


,event_id,event_class,waveform_id,has_waveform,n_primary_picks_raw,n_primary_picks_dedup_dropped,n_secondary_picks_in_window_raw,n_secondary_picks_dedup_dropped,n_picks_assigned_total,notes
0,pick_91060310,PICKS_ONLY,NaN,False,10,0,0,0,10,pick-only event (no waveform match/available)
1,pick_91061006,PICKS_ONLY,NaN,False,2,0,0,0,2,pick-only event (no waveform match/available)
2,pick_910610A6,PICKS_ONLY,NaN,False,2,0,0,0,2,pick-only event (no waveform match/available)
3,pick_910610B6,PICKS_ONLY,NaN,False,1,0,0,0,1,pick-only event (no waveform match/available)
4,pick_pinaug91_block0000,PICKS_ONLY,NaN,False,0,0,0,0,4,pick-only event (no waveform match/available)
